# Matrix-filtered neighbor generation

This notebook regenerates the left/right NCBI neighbor CSV and pocket candidate files, but keeps only neighbor genes that exist in the TPM gene-gene matrix.

Run modes:

- One disease: set `DISEASE_ID = "EFO_0000616"` and optionally set `FOLDER_NAME`.
- Whole folder: set `DISEASE_ID = None` and `FOLDER_NAME = "3_gene_related_disease"`.

After this notebook finishes, run `itterations.ipynb` to use the regenerated matrix-filtered pocket files.


In [163]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/miasmacbook/Desktop/KCL/6-months_project")
SUPPORT_PATH = PROJECT_ROOT / "script" / "disease_template_support.py"

# Set these before running.
# To run all diseases in a folder, use DISEASE_ID = None and FOLDER_NAME = "3_gene_related_disease".
DISEASE_ID = None
FOLDER_NAME = "123_gene_related_disease"

N_ROUNDS = 5000
RANDOM_SEED = 42
MAX_GENES_TO_CHANGE = 9
NEIGHBOR_COUNT_PER_SIDE = 5
ASSEMBLY = "GCF_000001405.40"

# If True, regenerate neighbor CSV, pocket candidates, matrix subset, rounds CSV, and graphs.
# If False, only regenerate the neighbor CSV and pocket candidates.
RUN_MODEL_AFTER_REGENERATING_NEIGHBORS = False


In [164]:
import csv
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("disease_template_support", SUPPORT_PATH)
template_support = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(template_support)


def discover_disease_ids(project_root: Path, folder_name: str | None, disease_id: str | None):
    if disease_id:
        return [disease_id]

    if not folder_name:
        raise ValueError("Set DISEASE_ID for one disease, or set FOLDER_NAME to run a whole folder.")

    folder = project_root / "separate_disease" / folder_name
    if not folder.exists():
        raise FileNotFoundError(f"Disease folder not found: {folder}")

    disease_ids = []
    for disease_dir in sorted(folder.iterdir()):
        if not disease_dir.is_dir():
            continue
        causal_csv = disease_dir / f"{disease_dir.name}.csv"
        if causal_csv.exists():
            disease_ids.append(disease_dir.name)

    if not disease_ids:
        raise FileNotFoundError(f"No disease folders with causal CSVs found under: {folder}")

    return disease_ids


def config_for_disease(disease_id: str):
    config = template_support.build_config(
        disease_id=disease_id,
        project_root=PROJECT_ROOT,
        n_rounds=N_ROUNDS,
        random_seed=RANDOM_SEED,
        max_genes_to_change=MAX_GENES_TO_CHANGE,
        neighbor_count_per_side=NEIGHBOR_COUNT_PER_SIDE,
        assembly=ASSEMBLY,
    )
    if FOLDER_NAME and config.pocket_dir.parent.name != FOLDER_NAME:
        raise ValueError(
            f"{disease_id} was found in {config.pocket_dir.parent.name}, not requested folder {FOLDER_NAME}."
        )
    return config


def count_neighbor_rows(path: Path) -> int:
    with path.open(newline="", encoding="utf-8") as handle:
        return sum(1 for _ in csv.DictReader(handle))


def validate_neighbor_csv_in_matrix(neighbor_csv: Path, matrix_gene_ids: set[str]):
    bad_ids = []
    with neighbor_csv.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            ids = [item.strip() for item in (row.get("neighbor_ensembl_ids") or "").split("|") if item.strip()]
            for gene_id in ids:
                if gene_id not in matrix_gene_ids:
                    bad_ids.append(gene_id)
    return bad_ids


def build_matrix_subset(required_gene_ids, source_matrix: Path, out_path: Path):
    required_gene_ids = {gene_id for gene_id in required_gene_ids if gene_id}
    if not required_gene_ids:
        raise ValueError("No matrix gene IDs were available for the subset.")

    header = __import__("pandas").read_csv(source_matrix, nrows=0)
    available_gene_ids = {str(column).strip() for column in header.columns[1:]}
    selected_gene_ids = sorted(required_gene_ids & available_gene_ids)
    if not selected_gene_ids:
        raise ValueError(f"None of the required genes were found in the source matrix: {source_matrix}")

    usecols = ["Name", *selected_gene_ids]
    chunks = []
    for chunk in __import__("pandas").read_csv(source_matrix, usecols=usecols, index_col=0, chunksize=4096):
        chunk.index = chunk.index.astype(str).str.strip()
        chunk.columns = chunk.columns.astype(str).str.strip()
        chunk = chunk.loc[chunk.index.intersection(selected_gene_ids)]
        if not chunk.empty:
            chunks.append(chunk)

    if not chunks:
        raise ValueError("No matrix rows were found for the selected genes.")

    subset = __import__("pandas").concat(chunks, axis=0)
    subset = subset.loc[selected_gene_ids, selected_gene_ids]
    subset.index.name = "Name"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    subset.to_csv(out_path)
    return subset


In [165]:
disease_ids = discover_disease_ids(PROJECT_ROOT, FOLDER_NAME, DISEASE_ID)
print(f"Diseases to process: {len(disease_ids)}")

summary_rows = []
for index, disease_id in enumerate(disease_ids, start=1):
    print(f"[{index}/{len(disease_ids)}] {disease_id}")
    config = config_for_disease(disease_id)

    # Always regenerate the neighbor CSV with the matrix filter.
    template_support.generate_neighbor_csv(config)

    causal_csv_path = template_support.resolve_input_file(config.causal_csv, "Causal CSV")
    neighbor_csv_path = template_support.resolve_input_file(config.neighbor_csv, "Neighbor CSV")
    causal_rows = template_support.read_rows(causal_csv_path)
    neighbor_rows = template_support.read_rows(neighbor_csv_path)
    matrix_gene_ids = template_support.load_matrix_gene_ids(config.source_matrix)

    bad_ids = validate_neighbor_csv_in_matrix(neighbor_csv_path, matrix_gene_ids)
    if bad_ids:
        raise ValueError(f"Neighbor CSV contains {len(bad_ids)} IDs not in matrix for {disease_id}: {bad_ids[:5]}")

    pockets = template_support.build_pockets(causal_rows, neighbor_rows, matrix_gene_ids=matrix_gene_ids)
    template_support.write_pocket_table(pockets, config.pocket_output_csv)

    required_gene_ids = template_support.collect_matrix_gene_ids(pockets)
    build_matrix_subset(required_gene_ids, config.source_matrix, config.matrix_path)

    if RUN_MODEL_AFTER_REGENERATING_NEIGHBORS:
        template_support.run_from_config(config)
        template_support.create_optional_plots(config)

    summary_rows.append(
        {
            "disease_id": disease_id,
            "folder_name": config.pocket_dir.parent.name,
            "neighbor_csv": config.neighbor_csv.as_posix(),
            "pocket_candidates_csv": config.pocket_output_csv.as_posix(),
            "matrix_subset_csv": config.matrix_path.as_posix(),
            "neighbor_rows": count_neighbor_rows(config.neighbor_csv),
            "pocket_candidate_genes": len(required_gene_ids),
            "all_neighbor_ids_in_matrix": True,
        }
    )

summary_df = __import__("pandas").DataFrame(summary_rows)
summary_path = PROJECT_ROOT / "matrix_filtered_neighbor_generation_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Summary CSV: {summary_path}")
summary_df


Diseases to process: 1
[1/1] EFO_0003756
Resolved seed genes: 123/123
Rows written: 1229
Neighbor CSV: /Users/miasmacbook/Desktop/KCL/6-months_project/separate_disease/123_gene_related_disease/EFO_0003756/EFO_0003756_left5_right5_ncbi_neighbors_with_ensembl.csv
Summary CSV: /Users/miasmacbook/Desktop/KCL/6-months_project/matrix_filtered_neighbor_generation_summary.csv


,disease_id,folder_name,neighbor_csv,pocket_candidates_csv,matrix_subset_csv,neighbor_rows,pocket_candidate_genes,all_neighbor_ids_in_matrix
0,EFO_0003756,123_gene_related_disease,/Users/miasmacbook/Desktop/KCL/6-months_projec...,/Users/miasmacbook/Desktop/KCL/6-months_projec...,/Users/miasmacbook/Desktop/KCL/6-months_projec...,1229,1320,True
